# 05 - Padrões de Gols

Média de gols por temporada, distribuição de placares, rodadas com mais gols.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)
df = df[df["ano"] >= 2003].copy()
df["total_gols"] = df["mandante_Placar"] + df["visitante_Placar"]

In [2]:
# Média de gols por temporada
gols_ano = df.groupby("ano")["total_gols"].mean().reset_index()
gols_ano.columns = ["ano", "media_gols"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=gols_ano["ano"], y=gols_ano["media_gols"],
    mode="lines+markers+text",
    text=gols_ano["media_gols"].round(2),
    textposition="top center",
    line=dict(color="#e74c3c", width=3),
    marker=dict(size=8),
    hovertemplate="%{x}: %{y:.2f} gols/jogo<extra></extra>"
))

fig.add_hline(y=df["total_gols"].mean(), line_dash="dash", line_color="gray",
              annotation_text=f"Média geral: {df['total_gols'].mean():.2f}",
              annotation_position="right")

fig.update_layout(
    title="Média de Gols por Partida por Temporada<br><sup>Brasileirão Série A (2003-presente)</sup>",
    xaxis_title="Ano", yaxis_title="Gols/Partida",
    template="plotly_white", width=900, height=500
)

fig.show()
_path = "../charts/media_gols.html"
fig.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Série temporal da média de gols por temporada: a variável oscila em torno da média histórica de ~2,56 gols/jogo, com baixa volatilidade interanual."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/media_gols.html")

Gráfico exportado: charts/media_gols.html


In [3]:
# Heatmap de placares (mandante x visitante)
max_gols = 6
df_heat = df.copy()
df_heat["gm"] = df_heat["mandante_Placar"].clip(upper=max_gols)
df_heat["gv"] = df_heat["visitante_Placar"].clip(upper=max_gols)

heatmap = pd.crosstab(df_heat["gm"], df_heat["gv"])
# Renomear eixos
labels_gols = [str(i) if i < max_gols else f"{max_gols}+" for i in range(max_gols + 1)]
heatmap.index = labels_gols
heatmap.columns = labels_gols

fig2 = px.imshow(
    heatmap,
    labels=dict(x="Gols Visitante", y="Gols Mandante", color="Partidas"),
    color_continuous_scale="YlOrRd",
    title="Distribuição de Placares<br><sup>Frequência de cada combinação de gols</sup>",
    text_auto=True
)
fig2.update_layout(template="plotly_white", width=600, height=550)

fig2.show()
_path = "../charts/heatmap_placares.html"
fig2.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Matriz de frequência de placares: a distribuição segue um padrão de Poisson bivariado, com alta concentração nos placares baixos (1x0, 1x1, 2x1) e decaimento exponencial."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/heatmap_placares.html")

Gráfico exportado: charts/heatmap_placares.html


In [4]:
# Média de gols por rodada
gols_rodada = df[df["rodata"] <= 38].groupby("rodata")["total_gols"].mean().reset_index()

fig3 = px.bar(gols_rodada, x="rodata", y="total_gols",
              title="Média de Gols por Rodada<br><sup>Rodadas iniciais e finais têm mais gols?</sup>",
              labels={"rodata": "Rodada", "total_gols": "Gols/Partida"},
              color="total_gols",
              color_continuous_scale="Reds")
fig3.update_layout(template="plotly_white", width=900, height=450, showlegend=False)

fig3.show()
_path = "../charts/gols_por_rodada.html"
fig3.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Média de gols por rodada: a distribuição é relativamente uniforme ao longo do campeonato, sem tendência clara de aumento nas rodadas finais."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/gols_por_rodada.html")

Gráfico exportado: charts/gols_por_rodada.html


In [5]:
# Top 10 maiores goleadas
df["diff"] = abs(df["mandante_Placar"] - df["visitante_Placar"])
top_goleadas = df.nlargest(10, ["total_gols", "diff"])[
    ["data", "mandante", "mandante_Placar", "visitante_Placar", "visitante", "ano"]
].copy()
top_goleadas["placar"] = top_goleadas.apply(
    lambda r: f"{r['mandante']} {int(r['mandante_Placar'])} x {int(r['visitante_Placar'])} {r['visitante']}", axis=1
)
print("Top 10 jogos com mais gols:")
for _, r in top_goleadas.iterrows():
    print(f"  {r['data'].strftime('%d/%m/%Y')} - {r['placar']}")

Top 10 jogos com mais gols:
  22/10/2003 - Bahia 4 x 7 Santos
  06/04/2003 - Vasco 6 x 4 Goias
  01/11/2006 - Athletico-PR 6 x 4 Vasco
  07/10/2023 - Goias 4 x 6 Bahia
  11/05/2008 - Portuguesa 5 x 5 Figueirense
  16/05/2004 - Criciuma 7 x 2 Goias
  27/07/2005 - Athletico-PR 7 x 2 Vasco
  12/05/2007 - Figueirense 3 x 6 Athletico-PR
  08/06/2017 - Chapecoense 3 x 6 Gremio
  06/08/2005 - Athletico-PR 5 x 4 Cruzeiro
